## 1. Load the tools
Imports are collected here so the notebook can be followed from top to bottom. `pandas` works with tables; scikit-learn supplies sampling, splitting, logistic regression, and metrics; seaborn and matplotlib draw charts.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.utils import resample
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
import seaborn as sn
import matplotlib.pyplot as plt


## 2. Load the customer data
Each row represents a bank customer. `subscribed` is the outcome we want to predict: **yes** or **no**. The remaining columns are customer and campaign information. The CSV must be in the notebook’s working folder.

In [ ]:
bank_df = pd.read_csv( 'bank.csv')
bank_df.head(5)

Inspect column names, data types, and missing values before modeling.

In [ ]:
bank_df.info()

Count the two outcomes. A model that predicts “no” for everyone could appear accurate when subscriptions are rare, so the class counts matter.

In [ ]:
bank_df.subscribed.value_counts()

## 3. Explore class imbalance and resampling
The following cells separate subscribers and non-subscribers, duplicate subscriber records with replacement until there are 2,000, combine the groups, then shuffle. This reduces class imbalance in the working data. **Teaching caveat:** because resampling happens before the train/test split in this original code, copies of one customer can occur in both sets. Treat the resulting test score as a demonstration, not an unbiased estimate of future performance. For a valid assessment, split the original customers first and resample the training set only.

In [ ]:
## Importing resample from *sklearn.utils* package.


Separate the two outcome groups: `yes` customers and `no` customers.

In [ ]:
# Separate the case of yes-subscribes and no-subscribes
bank_subscribed_no = bank_df[bank_df.subscribed == 'no']
bank_subscribed_yes = bank_df[bank_df.subscribed == 'yes']

Sample `yes` rows **with replacement**: a customer may be selected more than once. `n_samples=2000` is a chosen resampled count, not 2,000 new real customers.

In [ ]:
##Upsample the yes-subscribed cases.
df_minority_upsampled = resample(bank_subscribed_yes,replace=True, n_samples=2000) #2000

Put the 4,000 original `no` rows together with the resampled `yes` rows.

In [ ]:
# Combine majority class with upsampled minority class
new_bank_df = pd.concat([bank_subscribed_no, df_minority_upsampled])

Check the new total number of rows, then inspect how many belong to each class.

In [ ]:
len(new_bank_df)

In [ ]:
new_bank_df.subscribed.value_counts()

Shuffle the combined rows so the duplicated subscriber records are mixed with the other records.

In [ ]:
new_bank_df = shuffle(new_bank_df)

In [ ]:
print(new_bank_df)

## 4. Prepare inputs and answers
`X` holds input features; `Y` holds the actual answer. Remove `subscribed` from the feature list to prevent the model from seeing the answer while learning.

In [ ]:
# Assigning list of all column names in the DataFrame
X_features = list( new_bank_df.columns )

In [ ]:
# Remove the response variable from the list
X_features.remove( 'subscribed' )
X_features

Convert text categories into indicator columns using one-hot encoding. `drop_first=True` removes one indicator from each category set. The result is assigned to `X`.

In [ ]:
## get_dummies() will convert all the columns with data type as objects
encoded_bank_df = pd.get_dummies( new_bank_df[X_features], drop_first = True )
X = encoded_bank_df

In [ ]:
X

Convert the target from text to numbers: **yes → 1** and **no → 0**. Thus a predicted probability for class 1 means the probability of subscribing.

In [ ]:
# Encoding the subscribed column and assigning to Y
Y = new_bank_df.subscribed.map( lambda x: int( x == 'yes') )

## 5. Split and train
Reserve 30% of the *already resampled* rows for testing; train on 70%. `random_state=42` makes the split reproducible. The earlier resampling caveat still applies.

In [ ]:
## splitting training and test data
train_X, test_X, train_y, test_y = train_test_split( X,Y,test_size = 0.3,random_state = 42 )

Fit a logistic regression classifier. It learns a relationship between customer features and the probability of subscription. `fit` learns from the training rows.

In [ ]:
### logistic regression
## building the model
## Initializing the model
logit = LogisticRegression()
## Fitting the model with X and Y values of the dataset
logit.fit( train_X, train_y)


## 6. Predict classes
`predict(test_X)` returns **0 or 1** for each test row; 1 means predicted subscription. The next cell displays the predicted labels.

In [ ]:
## make prediction
pred_y = logit.predict(test_X)

In [ ]:
### predicting all the Y values for test_X
pred_y

Predict one example encoded customer. The numbers must correspond exactly to the columns and order in `X`; without that mapping, a standalone numeric list is hard to interpret as a real customer.

In [ ]:
## predicint
pred_single = logit.predict([[34,202,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1]])
pred_single

## 7. Evaluate the classifier
A confusion matrix compares actual and predicted labels. Here “subscribed” is class **1**. A false positive is a customer predicted to subscribe who did not; a false negative is a subscriber the model missed.

In [ ]:
## confusion matrix

In [ ]:
## Importing the metrics


Define the plotting function; the following cell calls it on test labels and predictions. Rows represent actual outcomes, columns predicted outcomes.

In [ ]:
## Defining the matrix to draw the confusion metrix from actual and predicted class labels
def draw_cm( actual, predicted ):
# Invoking confusion_matrix from metric package. The matrix will oriented as[1,0] i.e.
# the classes with label 1 will be reprensted the first row and 0 as secondrow
    cm = metrics.confusion_matrix( actual, predicted, [1,0] )
    ## Confustion will be plotted as heatmap for better visualization
    ## The lables are configured to better interpretation from the plot
    sn.heatmap(cm, annot=True, fmt='.2f',
    xticklabels = ["Subscribed", "Not Subscribed"] ,
    yticklabels = ["Subscribed", "Not Subscribed"] )
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()



In [ ]:
cm = draw_cm( test_y, pred_y )


### TP, TN, FP, and FN: read the four boxes

In this notebook, **positive = subscribed (1)**. Rows show the real answer; columns show the model's prediction.

| Actual | Predicted subscribed | Predicted did not subscribe |
|---|---|---|
| **Subscribed** | **TP:** correctly found a positive case | **FN:** missed a positive case |
| **Did not subscribe** | **FP:** raised a false alarm | **TN:** correctly found a negative case |

**True** means correct and **false** means incorrect. TP and FN refer to actual positive cases; FP and TN refer to actual negative cases. In the displayed heatmap, TP is top left, FN top right, FP bottom left, and TN bottom right.


### Precision: when the model says “subscribed,” can we trust it?

Suppose 10 customers are **predicted to subscribe**, and 8 actually are subscribed. Precision is **8 out of 10 = 80%**. The other 2 are false positives.

**Formula: precision = TP / (TP + FP).** It looks only at cases predicted positive. Use precision when uninterested customers contacted is costly. Higher precision means fewer false alarms among positive predictions.


### Recall: how many real positive cases did we find?

Suppose 10 customers actually are subscribed, and the model finds 7. Recall is **7 out of 10 = 70%**. The other 3 are false negatives.

**Formula: recall = TP / (TP + FN).** It looks only at actual positive cases. Use recall when missing potential subscribers is costly. Higher recall means fewer real positives missed.


### F1 score: one summary of precision and recall

**Formula: F1 = 2 × precision × recall / (precision + recall).** If precision is 80% and recall is 70%, F1 is about **74.7%**.

Use F1 when both false alarms and missed positive cases matter. If one of precision or recall is very low, F1 also falls. F1 does not tell you which type of mistake is more costly, so look at precision and recall separately too.


### Sensitivity: another name for recall

**Sensitivity = TP / (TP + FN) = recall.** It answers: *Of all customers who truly are subscribed, how many did the model find?*

If it finds 7 of 10 actual positive cases, sensitivity is **70%**. Use this name when discussing how often the model detects a real positive case. There is no separate sensitivity calculation to learn: it is the same number as recall for class 1.


### Specificity: how well do we recognize negative cases?

Suppose 10 customers actually are did not subscribe, and the model correctly identifies 9 of them. Specificity is **9 out of 10 = 90%**. The remaining one is a false positive.

**Formula: specificity = TN / (TN + FP).** It looks only at actual negative cases. Use it when you need to know how often the model avoids false alarms among customers who are did not subscribe. Sensitivity checks actual positives; specificity checks actual negatives.


**Using this notebook's matrix:** You can calculate precision, recall, F1, sensitivity, and specificity from its four displayed counts. The percentages above are teaching examples and may differ from the model's results.


## 8. Inspect predicted probabilities and ROC AUC
`predict_proba` returns two columns: probability of class **0** and probability of class **1**. ROC AUC evaluates how well the model ranks actual subscribers above non-subscribers across thresholds. The variable `chd_1` below is simply the existing notebook’s name for the class-1 probability.

In [ ]:

#ROC AUC Score

## Predicting the probability values for test cases
predict_proba_df = pd.DataFrame( logit.predict_proba( test_X ) )
predict_proba_df.head()
cm

Pair each test customer’s actual outcome with the model’s class-1 probability. `reset_index()` aligns the displayed rows with the probability table.

In [ ]:
## Initializing the DataFrame with actual class lables
test_results_df = pd.DataFrame( { 'actual': test_y } )
test_results_df = test_results_df.reset_index()
## Assigning the probability values for class label 1
test_results_df['chd_1'] = predict_proba_df.iloc[:,1:2]

In [ ]:
test_results_df.head(5)

Calculate ROC AUC from the actual outcomes and **probabilities**, not hard class predictions. Values nearer 1 indicate better ranking; about 0.5 is random ranking.

In [ ]:
# Passing actual class labels and the predicted probability values to compute ROC AUC score.
auc_score = metrics.roc_auc_score( test_results_df.actual, test_results_df.chd_1)
round( float( auc_score ), 2 )

Draw the ROC curve. Each point represents a different decision threshold and compares true positive rate with false positive rate.

In [ ]:
## The method takes the three following parameters
## model: the classification model
## test_X: X features of the test set
## test_y: actual labels of the test set
## Returns
## - ROC Auc Score
## - FPR and TPRs for different threshold values
def draw_roc_curve( model, test_X, test_y ):
    ## Creating and initializing a results DataFrame with actual labels
    test_results_df = pd.DataFrame( { 'actual': test_y } )
    test_results_df = test_results_df.reset_index()
    # predict the probabilities on the test set
    predict_proba_df = pd.DataFrame( model.predict_proba( test_X ) )
    ## selecting the probabilities that the test example belongs to class 1
    test_results_df['chd_1'] = predict_proba_df.iloc[:,1:2]
    ## Invoke roc_curve() to return the fpr, tpr and threshold values.
    ## threshold values contain values from 0.0 to 1.0
    fpr, tpr, thresholds = metrics.roc_curve( test_results_df.actual,
    test_results_df.chd_1,
    drop_intermediate = False )
    ## Getting the roc auc score by invoking metrics.roc_auc_score method
    auc_score = metrics.roc_auc_score( test_results_df.actual, test_results_df.chd_1 )
    ## Setting the size of the plot
    plt.figure(figsize=(8, 6))
    ## plotting the actual fpr and tpr values
    plt.plot( fpr, tpr, label='ROC curve (area = %0.2f)' % auc_score )
    ## plotting th diagnoal line from (0,1)
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    ## Setting labels and titles
    plt.xlabel('False Positive Rate or [1 - True Negative Rate]')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver operating characteristic example')
    plt.legend(loc="lower right")
    plt.show()
    return auc_score, fpr, tpr, thresholds

Run the ROC plot and inspect its AUC. Because duplicated customers may cross the split, do not present this value as an unbiased deployment result.

In [ ]:
## Invoking draw_roc_curve with the logistic regresson model
_, _, _, _ = draw_roc_curve( logit, test_X, test_y )

### How to understand ROC AUC, and how it differs from accuracy

The model gives each customer a **score: the estimated chance of subscribed**. A decision threshold turns that score into yes or no. For example, at a 0.5 threshold, scores above 0.5 are called positive. Moving the threshold changes how many positives we catch and how many false alarms we make.

**ROC curve:** Each point uses a different threshold. The vertical axis is **sensitivity/recall = TP / (TP + FN)**; the horizontal axis is **false positive rate = FP / (FP + TN) = 1 − specificity**. A curve closer to the **top left** catches more real positives with fewer false alarms. The diagonal represents random ranking.

**AUC** is the area under that curve. It measures how well the model ranks a randomly chosen positive case above a randomly chosen negative case: **0.5 ≈ random ranking; 1.0 = perfect ranking**. For example, an AUC of **0.80** means a positive case gets a higher score than a negative case about 80% of the time. AUC does not say that 80% of predictions are correct.

**Accuracy = (TP + TN) / (TP + TN + FP + FN)**: the share of correct **yes/no decisions at one chosen threshold**. ROC AUC summarizes **ranking across all thresholds**. For example, if 90 of 100 cases are negative, always predicting negative gives **90% accuracy** while finding **zero** positive cases. If its scores are also identical for everyone, its **AUC is 0.5**.

Read accuracy together with the confusion matrix, precision, recall, and specificity. Use ROC AUC to compare how well models rank cases before choosing a threshold. The sample AUC values here are illustrations, not measured results.


## Save the trained model as a pickle file

Pickle saves the fitted model so you can load it later without training it again. Run this cell **after** the model training cells. The saved model expects the same one-hot encoded input columns, in the same order as `X.columns`.

Only load pickle files from sources you trust.

In [ ]:
import pickle

with open("bank_subscription_model.pkl", "wb") as file:
    pickle.dump(logit, file)

print("Saved model to bank_subscription_model.pkl")